# ACSE — Adaptive Contextual Semantic Retention Engine
## Information-Theoretic Novelty Gating for Video Streams

This notebook demonstrates the full ACSE pipeline:
1. Synthetic dataset generation with ground-truth labels
2. Pipeline execution with per-frame scoring
3. Factor correlation analysis
4. Ablation studies
5. Latency benchmarks
6. Retention rate curves
7. Weight evolution
8. SHAP explanations
9. Memory entropy analysis
10. Error analysis

In [1]:
# Cell 1: Imports and setup
import sys, os
from pathlib import Path

# Ensure project root is on path
PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from collections import deque
import warnings
warnings.filterwarnings('ignore')

# ACSE core modules
from core.temporal_memory import TemporalGraphMemory
from core.novelty_engine import CognitiveNoveltyIndex
from core.event_decay import EventPersistenceDecay
from core.surprise_engine import SemanticSurpriseScore
from core.semantic_router import MultiFactorFusion
from core.explanation_engine import SHAPTraceExplainer
from core.adaptive_learning import OnlineBayesianUpdater
from core.decision_gate import AdaptiveThresholdGate

# Experiment modules
from experiments.dataset_builder import SyntheticVideoDatasetBuilder
from experiments.benchmark import ACSEBenchmark
from experiments.ablation import AblationStudy

# Plot style
plt.style.use('dark_background')
sns.set_palette('bright')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.15

print('ACSE modules loaded successfully.')
print(f'Project root: {PROJECT_ROOT}')

ACSE modules loaded successfully.
Project root: /home/bunnysingh/Music/acse_virat


In [2]:
# Cell 2: Build synthetic dataset
print('Building synthetic dataset...')
builder = SyntheticVideoDatasetBuilder(output_dir=str(PROJECT_ROOT / 'dataset'))
df = builder.build()
print(f'\nDataset shape: {df.shape}')
print(f'\nLabel distribution:\n{df["label"].value_counts()}')
print(f'\nScene types:\n{df["scene_type"].value_counts()}')
df.head(10)

Building synthetic dataset...
  Generating scene: STATIC_AMBIENT
  Generating scene: DYNAMIC_TRACKING
  Generating scene: MIXED_TRANSITION
  Dataset saved: /home/bunnysingh/Music/acse_virat/dataset/saml_frames.csv  (1500 rows)

Dataset shape: (1500, 9)

Label distribution:
label
0    1374
1     126
Name: count, dtype: int64

Scene types:
scene_type
STATIC_AMBIENT      500
DYNAMIC_TRACKING    500
MIXED_TRANSITION    500
Name: count, dtype: int64


,frame_idx,scene_type,label,cni,epd,sss,pig,raw,importance_score
0,0,STATIC_AMBIENT,0,0.0000,1.0000,0.5,0.0000,0.708,0.0000
1,1,STATIC_AMBIENT,0,0.0001,0.9071,0.5,0.0000,0.857,0.0000
2,2,STATIC_AMBIENT,0,0.0000,0.7808,0.5,0.0000,0.786,0.0000
3,3,STATIC_AMBIENT,0,0.0001,0.6720,0.5,0.0000,0.706,0.0110
4,4,STATIC_AMBIENT,0,0.0001,0.5784,0.5,0.0000,0.493,0.0169
5,5,STATIC_AMBIENT,0,0.0001,0.4978,0.5,0.0194,0.349,0.0183
6,6,STATIC_AMBIENT,0,0.0001,0.4285,0.5,0.0299,0.430,0.0289
7,7,STATIC_AMBIENT,0,0.0001,0.3688,0.5,0.0469,0.291,0.0236
8,8,STATIC_AMBIENT,0,0.0000,0.3174,0.5,0.0495,0.589,0.0539
9,9,STATIC_AMBIENT,0,0.0001,0.2732,0.5,0.0563,0.765,0.0773


In [3]:
# Cell 3: Show sample frames per scene type
sample_dir = PROJECT_ROOT / 'dataset' / 'sample_frames'
scene_types = ['static_ambient', 'dynamic_tracking', 'mixed_transition']

fig, axes = plt.subplots(3, 3, figsize=(14, 10))

for r, scene in enumerate(scene_types):
    samples = sorted(sample_dir.glob(f'{scene}_frame_*.png'))
    for c, img_path in enumerate(samples[:3]):
        img = cv2.imread(str(img_path))
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            axes[r, c].imshow(img)
        axes[r, c].set_title(f'{scene.upper()} — {img_path.stem}', fontsize=9)
        axes[r, c].axis('off')

plt.suptitle('Sample Frames per Scene Type', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'experiments' / 'results' / 'sample_frames.png'), dpi=150, bbox_inches='tight')
plt.show()

In [4]:
# Cell 4: Per-frame score timeline
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for ax, scene in zip(axes, df['scene_type'].unique()):
    mask = df['scene_type'] == scene
    scene_df = df[mask].reset_index(drop=True)
    
    ax.plot(scene_df.index, scene_df['importance_score'], color='#3b82f6', linewidth=0.8, alpha=0.9, label='Importance')
    ax.fill_between(scene_df.index, 0, scene_df['importance_score'], alpha=0.1, color='#3b82f6')
    
    # Mark ground truth important frames
    important = scene_df[scene_df['label'] == 1]
    ax.scatter(important.index, important['importance_score'], c='#10b981', s=8, zorder=5, label='GT Important')
    
    ax.set_ylabel('Score')
    ax.set_title(f'{scene}', fontsize=11, fontweight='bold')
    ax.legend(loc='upper right', fontsize=8)
    ax.set_ylim(-0.05, 1.05)

axes[-1].set_xlabel('Frame Index (within scene)')
plt.suptitle('ACSE Importance Score Timeline per Scene Type', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'experiments' / 'results' / 'score_timeline.png'), dpi=150, bbox_inches='tight')
plt.show()

In [5]:
# Cell 5: Factor correlation matrix
factor_cols = ['cni', 'epd', 'sss', 'pig', 'raw', 'importance_score', 'label']
corr = df[factor_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax,
            xticklabels=[c.upper() for c in factor_cols],
            yticklabels=[c.upper() for c in factor_cols])
ax.set_title('Factor Correlation Matrix (incl. Ground Truth Label)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'experiments' / 'results' / 'correlation_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

In [6]:
# Cell 6: Ablation study
print('Running ablation study...')
ablation = AblationStudy(results_dir=str(PROJECT_ROOT / 'experiments' / 'results'))
ablation_df = ablation.run(str(PROJECT_ROOT / 'dataset' / 'saml_frames.csv'))
print('\nAblation Results:')
ablation_df

Running ablation study...
  Running ablation: Full Pipeline
  Running ablation: Remove CNI
  Running ablation: Remove EPD
  Running ablation: Remove SSS
  Running ablation: Remove PIG
  Running ablation: Remove RAW
  Running ablation: Fixed Equal Weights
  Running ablation: No Temporal Memory
  Ablation results saved: /home/bunnysingh/Music/acse_virat/experiments/results/ablation_results.csv
  Ablation chart saved: /home/bunnysingh/Music/acse_virat/experiments/results/ablation_chart.png

Ablation Results:


,accuracy,f1,retention_rate,latency_s
ablation,,,,
Full Pipeline,0.7733,0.1282,0.1760,0.0935
Remove CNI,0.8387,0.0320,0.0827,0.0874
Remove EPD,0.7660,0.1894,0.2047,0.0898
Remove SSS,0.7820,0.0737,0.1513,0.1173
Remove PIG,0.7860,0.1058,0.1553,0.1245
Remove RAW,0.6560,0.1195,0.3067,0.0993
Fixed Equal Weights,0.8500,0.0741,0.0780,0.0850
No Temporal Memory,0.8407,0.0402,0.0820,0.0979


In [7]:
# Cell 7: Latency benchmark
print('Running latency benchmark...')
benchmark = ACSEBenchmark(results_dir=str(PROJECT_ROOT / 'experiments' / 'results'))
latency = benchmark.run_latency_benchmark(n_frames=300)

# Display as table
lat_df = pd.DataFrame(latency).T
print('\nLatency Report (ms):')
lat_df

Running latency benchmark...
  Latency report saved: /home/bunnysingh/Music/acse_virat/experiments/results/latency_report.json

Latency Report (ms):


,mean_ms,std_ms,p95_ms,p99_ms
memory_update,960.471,1232.190,3526.577,3953.297
cni_compute,0.495,0.541,0.611,2.917
epd_update,4.304,2.300,9.439,10.769
sss_update,51.807,12.939,75.724,91.332
fusion,24.108,8.768,34.046,56.597
gate,0.029,0.008,0.048,0.056
total,1041.219,1238.512,3608.243,4049.220


In [8]:
# Cell 8: Retention rate curve
print('Running retention benchmark...')
retention = benchmark.run_retention_benchmark(
    str(PROJECT_ROOT / 'dataset' / 'saml_frames.csv'),
    target_rates=[0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70]
)
ret_df = pd.DataFrame(retention).T
print('\nRetention Benchmark Results:')
ret_df

Running retention benchmark...
  Retention curve saved: /home/bunnysingh/Music/acse_virat/experiments/results/retention_curve.png

Retention Benchmark Results:


,target_rate,actual_rate,precision,recall
target_0.1,0.1,0.0647,0.1031,0.0794
target_0.2,0.2,0.1307,0.0867,0.1349
target_0.3,0.3,0.1760,0.0947,0.1984
target_0.4,0.4,0.2047,0.0814,0.1984
target_0.5,0.5,0.2147,0.0776,0.1984
target_0.6,0.6,0.2187,0.0762,0.1984
target_0.7,0.7,0.2187,0.0762,0.1984


In [9]:
# Cell 9: Weight evolution chart
# Re-run pipeline with adaptive learning to capture weight evolution
print('Running pipeline with adaptive weight tracking...')

memory = TemporalGraphMemory()
cni_engine = CognitiveNoveltyIndex()
epd_engine = EventPersistenceDecay()
sss_engine = SemanticSurpriseScore()
fusion = MultiFactorFusion()
updater = OnlineBayesianUpdater()
gate = AdaptiveThresholdGate(target_retention_rate=0.30)

# Generate fresh frames and run
rng = np.random.RandomState(42)
prev_frame = None
for i in range(500):
    frame = rng.randint(0, 255, (240, 320, 3), dtype=np.uint8)
    if i % 50 < 5:  # inject periodic events
        cv2.circle(frame, (160, 120), 40, (0, 255, 0), -1)
    
    desc = memory.update(frame, i, salience=0.5)
    cni = cni_engine.compute(desc, memory)
    epd_engine.update(frame, prev_frame, i)
    epd = epd_engine.compute_epd_score(i)
    sss_engine.update_predictor(desc)
    sss = sss_engine.compute(desc)
    
    fusion.set_weights(updater.get_weights())
    fused = fusion.fuse(cni, epd, sss, memory, i)
    result = gate.decide(fused['importance_score'], i)
    
    reward = 1.0 if result['decision'] == 'RETAIN' else -0.5
    updater.update(fused, reward)
    prev_frame = frame

# Plot weight evolution
weight_df = updater.get_weight_history()
fig, ax = plt.subplots(figsize=(12, 5))
colors = {'cni': '#3b82f6', 'epd': '#ef4444', 'sss': '#8b5cf6', 'pig': '#10b981'}
for col in weight_df.columns:
    ax.plot(weight_df.index, weight_df[col], label=col.upper(), color=colors.get(col, '#888'), linewidth=1.5)
ax.set_xlabel('Update Step')
ax.set_ylabel('Weight')
ax.set_title('Adaptive Weight Evolution Over Time', fontsize=13, fontweight='bold')
ax.legend()
ax.set_ylim(0, 0.7)
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'experiments' / 'results' / 'weight_evolution.png'), dpi=150, bbox_inches='tight')
plt.show()

Running pipeline with adaptive weight tracking...


In [10]:
# Cell 10: SHAP explanation for 5 example retained frames
print('Generating SHAP explanations for retained frames...\n')

memory2 = TemporalGraphMemory()
cni_engine2 = CognitiveNoveltyIndex()
epd_engine2 = EventPersistenceDecay()
sss_engine2 = SemanticSurpriseScore()
fusion2 = MultiFactorFusion()
explainer = SHAPTraceExplainer(fusion2)
gate2 = AdaptiveThresholdGate(target_retention_rate=0.30)

rng2 = np.random.RandomState(7)
retained_explanations = []
prev_frame2 = None

for i in range(200):
    frame = rng2.randint(0, 255, (240, 320, 3), dtype=np.uint8)
    if i % 30 == 0:
        cv2.rectangle(frame, (50, 50), (270, 190), (255, 100, 0), -1)
    
    desc = memory2.update(frame, i, salience=0.5)
    cni = cni_engine2.compute(desc, memory2)
    epd_engine2.update(frame, prev_frame2, i)
    epd = epd_engine2.compute_epd_score(i)
    sss_engine2.update_predictor(desc)
    sss = sss_engine2.compute(desc)
    fused = fusion2.fuse(cni, epd, sss, memory2, i)
    result = gate2.decide(fused['importance_score'], i)
    fused['decision'] = result['decision']
    
    if result['decision'] == 'RETAIN' and len(retained_explanations) < 5:
        expl = explainer.explain_decision(fused, i)
        retained_explanations.append(expl)
    
    prev_frame2 = frame

for expl in retained_explanations:
    print(f"Frame #{expl['frame_idx']}")
    print(f"  Decision: {expl['decision']}  Score: {expl['importance_score']:.4f}")
    print(f"  Dominant: {expl['dominant_factor'].upper()}")
    print(f"  Attribution: {expl['attribution']}")
    print(f"  Reason: {expl['human_reason']}")
    print()

Generating SHAP explanations for retained frames...

Frame #40
  Decision: RETAIN  Score: 0.4229
  Dominant: RAW
  Attribution: {'cni': 0.1356, 'epd': -0.000336, 'sss': 0.047695, 'pig': 0.002623, 'raw': -0.200838}
  Reason: Moderate resource-aware weight (RAW=0.68), contributing 52% of retention decision; High cognitive novelty index (CNI=1.00), contributing 35% of retention decision.

Frame #41
  Decision: RETAIN  Score: 0.3004
  Dominant: RAW
  Attribution: {'cni': 0.1214, 'epd': -0.000259, 'sss': -0.035718, 'pig': 0.002495, 'raw': -0.194472}
  Reason: Moderate resource-aware weight (RAW=0.61), contributing 55% of retention decision; High cognitive novelty index (CNI=1.00), contributing 34% of retention decision.

Frame #42
  Decision: RETAIN  Score: 0.4366
  Dominant: RAW
  Attribution: {'cni': 0.1398, 'epd': -0.000257, 'sss': 0.049967, 'pig': 0.002414, 'raw': -0.187996}
  Reason: Moderate resource-aware weight (RAW=0.70), contributing 49% of retention decision; High cognitive novel

In [11]:
# Cell 11: Memory entropy over time
print('Computing memory entropy over time...')

memory3 = TemporalGraphMemory()
entropies = []
rng3 = np.random.RandomState(99)

for i in range(500):
    # Create scenes with transitions
    if i < 150:
        # Static scene
        frame = np.full((240, 320, 3), 100, dtype=np.uint8)
        noise = rng3.normal(0, 3, frame.shape).astype(np.float32)
        frame = np.clip(frame.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    elif i < 300:
        # Dynamic scene
        frame = rng3.randint(0, 255, (240, 320, 3), dtype=np.uint8)
    else:
        # Mixed scene
        frame = np.full((240, 320, 3), 60, dtype=np.uint8)
        if i % 20 < 10:
            cv2.circle(frame, (160, 120), 50 + (i % 30), (0, 200, 255), -1)

    memory3.update(frame, i, salience=0.5)
    entropies.append(memory3.get_memory_entropy())

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(range(len(entropies)), entropies, color='#06b6d4', linewidth=1.2)
ax.fill_between(range(len(entropies)), 0, entropies, alpha=0.1, color='#06b6d4')
ax.axvline(150, color='#f59e0b', linestyle='--', alpha=0.5, label='Static→Dynamic')
ax.axvline(300, color='#ec4899', linestyle='--', alpha=0.5, label='Dynamic→Mixed')
ax.set_xlabel('Frame Index')
ax.set_ylabel('Memory Entropy (bits)')
ax.set_title('Memory Entropy Over Time — Scene Transitions Cause Diversity Spikes', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'experiments' / 'results' / 'memory_entropy.png'), dpi=150, bbox_inches='tight')
plt.show()

Computing memory entropy over time...


In [12]:
# Cell 12: Error analysis — false positives and false negatives
print('Error analysis: examining misclassifications...')

gate_err = AdaptiveThresholdGate(target_retention_rate=0.30)
df['predicted'] = df['importance_score'].apply(lambda s: gate_err.decide(s, 0)['decision'])
df['pred_label'] = df['predicted'].apply(lambda d: 1 if d == 'RETAIN' else 0)

# False positives: predicted RETAIN but label=0
fp = df[(df['pred_label'] == 1) & (df['label'] == 0)]
# False negatives: predicted DISCARD but label=1
fn = df[(df['pred_label'] == 0) & (df['label'] == 1)]

print(f'\nFalse Positives: {len(fp)} frames retained incorrectly')
print(f'False Negatives: {len(fn)} important frames missed')
print(f'\nFP factor means:')
if len(fp) > 0:
    print(fp[['cni', 'epd', 'sss', 'pig', 'raw']].mean().round(3).to_string())
print(f'\nFN factor means:')
if len(fn) > 0:
    print(fn[['cni', 'epd', 'sss', 'pig', 'raw']].mean().round(3).to_string())

# Plot distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if len(fp) > 0:
    fp[['cni', 'epd', 'sss', 'pig']].plot.box(ax=axes[0])
    axes[0].set_title('False Positives — Factor Distributions', fontweight='bold')
    axes[0].set_ylabel('Value')
else:
    axes[0].text(0.5, 0.5, 'No false positives', ha='center', va='center', transform=axes[0].transAxes)
    axes[0].set_title('False Positives')

if len(fn) > 0:
    fn[['cni', 'epd', 'sss', 'pig']].plot.box(ax=axes[1])
    axes[1].set_title('False Negatives — Factor Distributions', fontweight='bold')
    axes[1].set_ylabel('Value')
else:
    axes[1].text(0.5, 0.5, 'No false negatives', ha='center', va='center', transform=axes[1].transAxes)
    axes[1].set_title('False Negatives')

plt.suptitle('Error Analysis: What Drives Misclassifications?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'experiments' / 'results' / 'error_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

# Cleanup temp columns
df.drop(columns=['predicted', 'pred_label'], inplace=True)

Error analysis: examining misclassifications...

False Positives: 239 frames retained incorrectly
False Negatives: 101 important frames missed

FP factor means:
cni    0.642
epd    0.089
sss    0.658
pig    0.173
raw    0.470

FN factor means:
cni    0.632
epd    0.487
sss    0.483
pig    0.152
raw    0.250


In [15]:
# Cell 13: Single-frame inference demo
print('=== Single-Frame Inference Demo ===')
print()

# Create a test frame with an object
demo_frame = np.full((240, 320, 3), 80, dtype=np.uint8)
cv2.circle(demo_frame, (200, 100), 35, (0, 220, 180), -1)
cv2.rectangle(demo_frame, (40, 150), (120, 220), (220, 60, 60), -1)

# Fresh pipeline
demo_memory = TemporalGraphMemory()
demo_cni = CognitiveNoveltyIndex()
demo_epd = EventPersistenceDecay()
demo_sss = SemanticSurpriseScore()
demo_fusion = MultiFactorFusion()
demo_explainer = SHAPTraceExplainer(demo_fusion)
demo_gate = AdaptiveThresholdGate(target_retention_rate=0.30)

# Warm up with a blank frame
blank = np.full((240, 320, 3), 80, dtype=np.uint8)
for warmup_i in range(15):
    noise_frame = blank.copy()
    noise = np.random.normal(0, 2, noise_frame.shape).astype(np.float32)
    noise_frame = np.clip(noise_frame.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    desc_w = demo_memory.update(noise_frame, warmup_i, salience=0.5)
    demo_sss.update_predictor(desc_w)
    demo_sss.compute(desc_w)
    demo_epd.update(noise_frame, blank if warmup_i > 0 else None, warmup_i)

# Now process the test frame
frame_idx = 15
desc = demo_memory.update(demo_frame, frame_idx, salience=0.5)
cni_val = demo_cni.compute(desc, demo_memory)
demo_epd.update(demo_frame, blank, frame_idx)
epd_val = demo_epd.compute_epd_score(frame_idx)
demo_sss.update_predictor(desc)
sss_val = demo_sss.compute(desc)

fused = demo_fusion.fuse(cni_val, epd_val, sss_val, demo_memory, frame_idx)
gate_result = demo_gate.decide(fused['importance_score'], frame_idx)
fused['decision'] = gate_result['decision']

explanation = demo_explainer.explain_decision(fused, frame_idx)

print(f"Frame #{frame_idx} Inference Results:")
print(f"  CNI  = {cni_val:.4f}  (Cognitive Novelty Index)")
print(f"  EPD  = {epd_val:.4f}  (Event Persistence Decay)")
print(f"  SSS  = {sss_val:.4f}  (Semantic Surprise Score)")
print(f"  PIG  = {fused['pig']:.4f}  (Predictive Information Gain)")
print(f"  RAW  = {fused['raw']:.4f}  (Resource-Aware Weight)")
print(f"  ─────────────────────────────────")
print(f"  Importance Score = {fused['importance_score']:.4f}")
print(f"  Threshold        = {gate_result['threshold']:.4f}")
print(f"  Decision         = {gate_result['decision']}")
print(f"  ─────────────────────────────────")
print(f"  Dominant Factor  = {explanation['dominant_factor'].upper()}")
print(f"  Reason: {explanation['human_reason']}")
print()
print('Attribution breakdown:')
for factor, val in explanation['attribution'].items():
    bar = '█' * int(abs(val) * 50)
    sign = '+' if val >= 0 else '-'
    print(f"  {factor.upper():5s} {sign}{abs(val):.4f}  {bar}")

=== Single-Frame Inference Demo ===

Frame #15 Inference Results:
  CNI  = 0.0000  (Cognitive Novelty Index)
  EPD  = 0.6190  (Event Persistence Decay)
  SSS  = 0.2959  (Semantic Surprise Score)
  PIG  = 0.0513  (Predictive Information Gain)
  RAW  = 0.8130  (Resource-Aware Weight)
  ─────────────────────────────────
  Importance Score = 0.0000
  Threshold        = 0.5000
  Decision         = DISCARD
  ─────────────────────────────────
  Dominant Factor  = CNI
  Reason: Low cognitive novelty index (CNI=0.00), contributing 58% of retention decision; High event persistence decay (EPD=0.62), contributing 32% of retention decision.

Attribution breakdown:
  CNI   -0.1383  ██████
  EPD   -0.0764  ███
  SSS   -0.0255  █
  PIG   +0.0000  
  RAW   +0.0000  
